## Lab 1: Data Representation

Dianne Yumol

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import io
import math
import warnings

# Optional libraries for thresholding
from scipy.stats import genpareto
from sklearn.mixture import GaussianMixture
try:
    from kneed import KneeLocator
    _HAS_KNEED = True
except Exception:
    _HAS_KNEED = False

In [2]:
class BaseScaler:
    def fit(self, X):
        raise NotImplementedError
    def transform(self, X):
        raise NotImplementedError
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

class SimpleMinMaxScaler(BaseScaler):
    def __init__(self, feature_range=(0.0, 1.0)):
        self._min = None
        self._max = None
        self._range = feature_range

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self._min = np.nanmin(X, axis=0)
        self._max = np.nanmax(X, axis=0)
        self._denom = np.where(self._max - self._min == 0, 1.0, (self._max - self._min))

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        scale = (self._range[1] - self._range[0]) / self._denom
        return self._range[0] + (X - self._min) * scale

class SimpleZScoreScaler(BaseScaler):
    def __init__(self):
        self._mean = None
        self._std = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self._mean = np.nanmean(X, axis=0)
        self._std  = np.nanstd(X, axis=0)
        self._std = np.where(self._std == 0, 1.0, self._std)

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self._mean) / self._std

class SimpleRobustScaler(BaseScaler):
    def __init__(self):
        self._median = None
        self._iqr = None 

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        q75, q25 = np.percentile(X, [75, 25], axis=0)
        self._median = np.nanmedian(X, axis=0)
        self._iqr = q75 - q25   # Save as _iqr

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        iqr_safe = np.where(self._iqr == 0, 1.0, self._iqr)  # Use _iqr consistently
        return (X - self._median) / iqr_safe

In [3]:
class Imputer:
    def __init__(self, strategy="mean", columns=None):
        assert strategy in ("mean", "median", "most_frequent")
        self.strategy = strategy
        self.columns = columns
        self.statistics_ = {}

    def fit(self, df):
        if self.columns is None:
            if self.strategy in ("mean", "median"):
                cols = df.select_dtypes(include=[np.number]).columns.tolist()
            else:
                cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
        else:
            cols = list(self.columns)

        for col in cols:
            if self.strategy in ("mean", "median"):
                s = pd.to_numeric(df[col], errors='coerce')
                if self.strategy == "mean":
                    stat = s.mean(skipna=True)
                else:
                    stat = s.median(skipna=True)
                if pd.isna(stat):
                    mode = df[col].mode(dropna=True)
                    stat = mode.iloc[0] if not mode.empty else 0.0
                self.statistics_[col] = stat
            else: 
                mode = df[col].mode(dropna=True)
                self.statistics_[col] = mode.iloc[0] if not mode.empty else np.nan
        return self

    def transform(self, df):
        df = df.copy()
        for col, val in self.statistics_.items():
            df[col] = df[col].fillna(val)
        return df

    def fit_transform(self, df):
        return self.fit(df).transform(df)

In [ ]:
def _gpd_quantiles(sigma: float, xi: float, probs: np.ndarray) -> np.ndarray:
    if abs(xi) < 1e-12:
        return -sigma * np.log(1 - probs)
    else:
        return (sigma / xi) * (np.power(1 - probs, -xi) - 1.0)

def eqd_select_threshold_on_variances(
    variances: np.ndarray,
    candidate_percentiles: np.ndarray = None,
    B: int = 100,
    m: int = 200,
    min_exceedances: int = 5,
    verbose: bool = True
):
    variances = np.array(variances, dtype=float)
    variances = variances[np.isfinite(variances)]
    if variances.size == 0:
        raise ValueError("Empty variances array.")

    if candidate_percentiles is None:
        candidate_percentiles = np.linspace(0.0, 95.0, 20)
    candidate_thresholds = np.percentile(variances, candidate_percentiles)
    candidate_thresholds = np.unique(np.sort(candidate_thresholds))

    pj = np.linspace(1.0 / (m + 1), m / (m + 1), m)
    dE_vals = []
    details = {}

    for u in candidate_thresholds:
        mask = variances > u
        excesses = variances[mask] - u
        nu = len(excesses)
        if nu < min_exceedances:
            dE_vals.append(np.inf)
            details[u] = {"nu": nu, "reason": "too_few_exceedances"}
            continue
        db_list = []
        for b in range(B):
            xb = np.random.choice(excesses, size=nu, replace=True)
            try:
                c, loc, scale = genpareto.fit(xb, floc=0)
                xi = float(c); sigma = float(scale)
                if sigma <= 0 or not np.isfinite(xi):
                    db_list.append(np.inf)
                    continue
            except Exception:
                db_list.append(np.inf)
                continue
            q_model = _gpd_quantiles(sigma, xi, pj)
            q_sample = np.quantile(xb, pj)
            db = np.mean(np.abs(q_model - q_sample))
            db_list.append(db)
        finite_vals = [v for v in db_list if np.isfinite(v)]
        dE = np.mean(finite_vals) if finite_vals else np.inf
        dE_vals.append(dE)
        details[u] = {"nu": nu, "dE": dE, "db_samples_len": len(db_list)}

    dE_vals = np.array(dE_vals)
    if np.all(~np.isfinite(dE_vals)):
        # fallback: median variance
        selected = float(np.median(variances))
        sel_percentile = 50.0
    else:
        best_idx = int(np.nanargmin(dE_vals))
        selected = float(candidate_thresholds[best_idx])
        # map candidate threshold back to percentile approximately
        sel_percentile = float(np.percentile(variances, np.linspace(0.0, 100.0, len(candidate_thresholds))[best_idx]))
    if verbose:
        print(f"[EQD] chosen threshold = {selected:.6g} (rough percentile {sel_percentile:.2f})")
    return {"selected_threshold": selected, "selected_percentile": sel_percentile, "candidate_thresholds": candidate_thresholds, "dE_values": dE_vals, "details": details}

def otsu_threshold_from_variances(variances: np.ndarray) -> float:
    v = np.array(variances, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return float(np.median(v))
    hist, bin_edges = np.histogram(v, bins='auto')
    probs = hist.astype(float) / np.sum(hist)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    omega0 = np.cumsum(probs)
    mu0 = np.cumsum(probs * bin_centers) / (omega0 + 1e-12)
    muT = np.sum(probs * bin_centers)
    omega1 = 1.0 - omega0
    mu1 = (muT - omega0 * mu0) / (omega1 + 1e-12)
    sigma_b2 = omega0 * omega1 * (mu0 - mu1) ** 2
    idx = int(np.nanargmax(sigma_b2))
    return float(bin_centers[idx])

def gmm_threshold_from_variances(variances: np.ndarray, random_state: int = 0) -> float:
    v = np.array(variances, dtype=float).reshape(-1, 1)
    v = v[np.isfinite(v).reshape(-1,)]
    if len(v) < 2:
        return float(np.median(v))
    vt = np.log1p(v)
    gmm = GaussianMixture(n_components=2, random_state=random_state)
    try:
        gmm.fit(vt)
        grid = np.linspace(vt.min(), vt.max(), 300)
        resp = gmm.predict_proba(grid.reshape(-1,1))
        diff = resp[:, 0] - resp[:, 1]
        ix = int(np.argmin(np.abs(diff)))
        thr_vt = grid[ix]
        return float(np.expm1(thr_vt))
    except Exception:
        return float(np.median(v))

def elbow_threshold_from_variances(variances: np.ndarray) -> float:
    v = np.sort(np.array(variances, dtype=float))
    v = v[np.isfinite(v)]
    if len(v) < 3:
        return float(np.median(v))
    y = v[::-1]  # descending to form L-shape
    x = np.arange(1, len(y)+1)
    if _HAS_KNEED:
        try:
            k = KneeLocator(x, y, curve='convex', direction='decreasing', interp_method='polynomial')
            if k.knee is not None:
                idx = int(k.knee - 1)
                return float(y[idx])
        except Exception:
            pass
    # fallback: max second derivative index
    sec = np.abs(np.diff(y, n=2))
    idx = int(np.argmax(sec)) + 1
    return float(y[idx])

In [ ]:
def apply_scaler_and_auto_variance_prune(df_full: pd.DataFrame, numeric_cols: list, scaler: BaseScaler,
                                         method: str = "eqd", eqd_params: dict = None, verbose: bool = True):
    # Returns the scaled combined DataFrame, pruned_df
    df = df_full.copy().reset_index(drop=True)
    Xnum = df[numeric_cols].to_numpy(dtype=float)
    scaler.fit(Xnum)
    Xs = scaler.transform(Xnum)
    scaled_numeric_df = pd.DataFrame(Xs, columns=numeric_cols, index=df.index)

    other_cols = [c for c in df.columns if c not in numeric_cols]
    combined = pd.concat([scaled_numeric_df, df[other_cols].reset_index(drop=True)], axis=1)

    numeric_combined = combined.select_dtypes(include=[np.number])
    variances = numeric_combined.var(axis=0, ddof=0)
    var_values = variances.values
    col_names = variances.index.tolist()

    if method == "eqd":
        params = {} if eqd_params is None else dict(eqd_params)
        params.setdefault("B", 100)
        params.setdefault("m", 200)
        params.setdefault("candidate_percentiles", np.linspace(0.0, 95.0, 20))
        params.setdefault("min_exceedances", max(3, int(0.05 * len(var_values))))
        info = eqd_select_threshold_on_variances(var_values, candidate_percentiles=params["candidate_percentiles"],
                                                B=params["B"], m=params["m"], min_exceedances=params["min_exceedances"], verbose=verbose)
        thr = info["selected_threshold"]
        selector_info = info
    elif method == "otsu":
        thr = otsu_threshold_from_variances(var_values)
        selector_info = {"method": "otsu"}
        if verbose:
            print(f"[Otsu] threshold = {thr:.6g}")
    elif method == "gmm":
        thr = gmm_threshold_from_variances(var_values)
        selector_info = {"method": "gmm"}
        if verbose:
            print(f"[GMM] threshold = {thr:.6g}")
    elif method == "elbow":
        thr = elbow_threshold_from_variances(var_values)
        selector_info = {"method": "elbow"}
        if verbose:
            print(f"[Elbow] threshold = {thr:.6g}")
    else:
        raise ValueError("Unknown method")

    kept_mask = variances >= thr
    kept_numeric_cols = [c for c, keep in zip(col_names, kept_mask) if keep]
    removed_numeric_cols = [c for c, keep in zip(col_names, kept_mask) if not keep]

    pruned_cols = list(kept_numeric_cols)
    if 'Churn' in combined.columns:
        # ensure Churn is included (and numeric)
        pruned_cols.append('Churn')
    pruned_df = combined.loc[:, pruned_cols].copy()

    if verbose:
        print(f"Threshold selected: {thr:.6g} -> kept {len(kept_numeric_cols)}/{len(col_names)} numeric features; removed {len(removed_numeric_cols)}")

    return {
        "scaled_df": combined,
        "pruned_df": pruned_df,
        "kept_columns": kept_numeric_cols,
        "removed_columns": removed_numeric_cols,
        "selected_threshold": thr,
        "variances": variances,
        "selector_info": selector_info
    }

In [6]:
class TimeSeriesDataset(Dataset):
    def __init__(self, df_numeric, window_size, num_outputs, stride=1,
                 feature_cols=None, target_cols=None, transform=None, target_transform=None):
        if window_size <= 0 or num_outputs <= 0 or stride <= 0:
            raise ValueError("window_size, num_outputs and stride must be positive integers.")
        self.window_size = int(window_size)
        self.num_outputs = int(num_outputs)
        self.stride = int(stride)
        self.transform = transform
        self.target_transform = target_transform

        if isinstance(df_numeric, pd.DataFrame):
            if feature_cols is None:
                feature_cols = list(df_numeric.columns)
            if target_cols is None:
                target_cols = feature_cols
            self.feature_cols = list(feature_cols)
            self.target_cols = list(target_cols)
            self.features = df_numeric[self.feature_cols].to_numpy(dtype=np.float32)
            self.targets = df_numeric[self.target_cols].to_numpy(dtype=np.float32)
        else:
            n_cols = df_numeric.shape[1]
            self.feature_cols = list(range(n_cols))
            self.target_cols = self.feature_cols
            self.features = np.asarray(df_numeric, dtype=np.float32)
            self.targets = self.features

        n_rows = len(self.features)
        last_start = n_rows - self.window_size - self.num_outputs
        if last_start < 0:
            self.start_indices = []
        else:
            self.start_indices = list(range(0, last_start + 1, self.stride))

    def __len__(self):
        return len(self.start_indices)

    def __getitem__(self, idx):
        if isinstance(idx, torch.Tensor):
            idx = idx.item()
        start = self.start_indices[idx]
        x_np = self.features[start : start + self.window_size]
        y_np = self.targets[start + self.window_size : start + self.window_size + self.num_outputs]
        x = torch.from_numpy(x_np).float()
        y = torch.from_numpy(y_np).float()
        if self.transform is not None:
            x = self.transform(x)
        if self.target_transform is not None:
            y = self.target_transform(y)
        return x, y


In [ ]:
def main():
    df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

    # Ensure TotalCharges numeric
    if 'TotalCharges' in df.columns:
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    
    # Drop identifier column(s) before one-hot encoding to avoid non-numeric columns later
    if 'customerID' in df.columns:
        df = df.drop(columns=['customerID'])

    exclude = ['Churn']
    cat_cols = [c for c in df.select_dtypes(include=['object','category','bool']).columns if c not in exclude]

    # Impute numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_imputer = Imputer(strategy="median", columns=numeric_cols)
    df = num_imputer.fit_transform(df)

    # Impute categorical columns
    if cat_cols:
        cat_imputer = Imputer(strategy="most_frequent", columns=cat_cols)
        df = cat_imputer.fit_transform(df)

    # One-hot encode
    df_ohe = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

    # Final safety-fill if any NaNs left (numeric)
    if df_ohe.isna().values.any():
        df_ohe.fillna(df_ohe.median(numeric_only=True), inplace=True)

    # Choose numeric columns again (they may include newly converted numeric)
    numeric_cols = [c for c in df_ohe.columns if df_ohe[c].dtype.kind in "fi" and c != "Churn"]

    # Prepare scalers
    scalers = {
        "minmax": SimpleMinMaxScaler(feature_range=(0.0, 1.0)),
        "zscore": SimpleZScoreScaler(),
        "robust": SimpleRobustScaler()
    }

    # For each scaler, produce pruned dataset via automated variance thresholding
    all_results = {}
    for name, scaler in scalers.items():
        print("\n--- Processing with scaler:", name)
        res = apply_scaler_and_auto_variance_prune(df_ohe, numeric_cols=numeric_cols, scaler=scaler,
                                                   method="eqd", eqd_params={"B":100, "m":200}, verbose=True)
        all_results[name] = res
        # Print summary
        print(f"Scaler={name}: kept {len(res['kept_columns'])} features; removed {len(res['removed_columns'])}.")
        # Show some of the removed columns if any
        if res['removed_columns']:
            print("Removed example columns:", res['removed_columns'][:8])

    for name, r in all_results.items():
        outfile = f"telco_pruned_{name}.csv"
        r["pruned_df"].to_csv(outfile, index=False)
        print(f"Saved pruned dataset for scaler {name} to {outfile}")

    # Load electric production CSV
    try:
        df2 = pd.read_csv("electric_production.csv")
    except FileNotFoundError:
        print("electric_production.csv not found in current dir; skipping timeseries demo.")
        return

    numeric_df = df2.copy()
    for c in numeric_df.columns:
        numeric_df[c] = pd.to_numeric(numeric_df[c], errors="coerce")
    numeric_df = numeric_df.loc[:, ~numeric_df.isna().all()]

    # Create dataset and loader
    dataset = TimeSeriesDataset(numeric_df, window_size=12, num_outputs=3, stride=1)
    loader = DataLoader(dataset, batch_size=8, shuffle=False)
    try:
        batch_x, batch_y = next(iter(loader))
        print("Example batch shapes:", batch_x.shape, batch_y.shape)
    except StopIteration:
        print("TimeSeriesDataset is empty (not enough rows for window+output).")

    # Print first few samples
    for i in range(min(5, len(dataset))):
        x, y = dataset[i]
        print(f"\nSample {i}:")
        # show flattened for readability if single-feature
        print(" x:", x.squeeze().tolist())
        print(" y:", y.squeeze().tolist())

if __name__ == "__main__":
    main()


--- Processing with scaler: minmax
[EQD] chosen threshold = 0.249972 (rough percentile 0.25)
Threshold selected: 0.249972 -> kept 3/45 numeric features; removed 42
Scaler=minmax: kept 3 features; removed 42.
Removed example columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes']

--- Processing with scaler: zscore
[EQD] chosen threshold = 1 (rough percentile 1.00)
Threshold selected: 1 -> kept 12/45 numeric features; removed 33
Scaler=zscore: kept 12 features; removed 33.
Removed example columns: ['tenure', 'MonthlyCharges', 'TotalCharges', 'Partner_Yes', 'Dependents_No', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No']

--- Processing with scaler: robust
[EQD] chosen threshold = 0.236788 (rough percentile 0.23)
Threshold selected: 0.236788 -> kept 20/45 numeric features; removed 25
Scaler=robust: kept 20 features; removed 25.
Removed example columns: ['SeniorCitizen', 'Dependents_No', 'Dependents